In [16]:
import numpy as np
import xarray as xr
import metpy.calc as mpcalc
import time
import os
import glob
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


FIG_SAVE_DIR = "./figures/divergence/"

# 创建必要的目录
for d in [ FIG_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)


EXPERIMENTS = ['CNTL', 'P4K', '4CO2']

In [17]:
def _ocean(ds):
    fraction = xr.open_dataarray(r'../processed_data/land_mask_2deg.nc')
    return fraction == 0
ocean_mask = _ocean(None)
from dask.distributed import Client

In [18]:
try:
    phalf_cntl = xr.open_dataarray('../3D_data/cntl/phalf_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    phalf_p4k = xr.open_dataarray('../3D_data/p4k/phalf_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    phalf_4co2 = xr.open_dataarray('../3D_data/4co2/phalf_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
except Exception as e:
    print(f"  警告: 未找到气压数据 ({e})")
    print("  将使用默认设置")
    phalf_cntl = None
    phalf_p4k = None
    phalf_4co2 = None
    
try:
    pfull_cntl = xr.open_dataarray('../3D_data/cntl/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    pfull_p4k = xr.open_dataarray('../3D_data/p4k/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    pfull_4co2 = xr.open_dataarray('../3D_data/4co2/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
except Exception as e:
    print(f"  警告: 未找到气压数据 ({e})")
    print("  将使用默认设置")
    pfull_cntl = None
    pfull_p4k = None
    pfull_4co2 = None   
try:
    ua_cntl = xr.open_dataarray('../3D_data/cntl/ua_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    ua_p4k = xr.open_dataarray('../3D_data/p4k/ua_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    ua_4co2 = xr.open_dataarray('../3D_data/4co2/ua_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
except Exception as e:
    print(f"  警告: 未找到ua数据 ({e})")
    print("  将使用默认设置")
    ua_cntl = None
    ua_p4k = None
    ua_4co2 = None
try:
    va_cntl = xr.open_dataarray('../3D_data/cntl/va_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    va_p4k = xr.open_dataarray('../3D_data/p4k/va_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
    va_4co2 = xr.open_dataarray('../3D_data/4co2/va_all_levels.nc', chunks={'time': 10}).where(ocean_mask, drop=True)
except Exception as e:
    print(f"  警告: 未找到va数据 ({e})")
    print("  将使用默认设置")
    va_cntl = None
    va_p4k = None
    va_4co2 = None

In [19]:
# Dask 并行配置
from dask.distributed import Client, LocalCluster
import multiprocessing

USE_SLURM = False  # True: SLURM集群, False: 本地多核

if USE_SLURM:
    from dask_jobqueue import SLURMCluster
    cluster = SLURMCluster(
        cores=128, processes=16, memory="240GB", walltime="04:00:00",
        account="mh1498", queue="shared", interface="ib0",
        local_directory="/work/mh1498/m301257/dask-scratch-space"
    )
    cluster.scale(jobs=3)
else:
    n_workers = max(4, multiprocessing.cpu_count() // 2)
    cluster = LocalCluster(n_workers=n_workers, threads_per_worker=2, memory_limit='8GB')

client = Client(cluster)
print(f"✅ Dask 集群启动 | Workers: {n_workers if not USE_SLURM else '48'} | Dashboard: {client.dashboard_link}")

/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35245 instead
  warnings.warn(


✅ Dask 集群启动 | Workers: 128 | Dashboard: http://127.0.0.1:35245/status


In [20]:
# 核心处理函数
def compute_grid_spacing(lat, lon):
    """计算网格间距"""
    R = 6.371e6
    xlon, ylat = np.meshgrid(lon, lat)
    dx = R * np.cos(ylat * np.pi / 180) * np.gradient(xlon, axis=1) * np.pi / 180
    dy = R * np.gradient(ylat, axis=0) * np.pi / 180
    return dx, dy

def numpy_divergence(u, v, dx, dy):
    """计算散度"""
    u = u.values if isinstance(u, xr.DataArray) else u
    v = v.values if isinstance(v, xr.DataArray) else v
    dx_exp = dx[np.newaxis, :, :] if u.ndim == 3 else dx
    dy_exp = dy[np.newaxis, :, :] if u.ndim == 3 else dy
    return np.gradient(u, axis=-1) / dx_exp + np.gradient(v, axis=-2) / dy_exp

In [21]:
def hacky_plvl_interpolation(data_var, data_pfull, plvl_target, unit='hPa', height_name='level'):
    """气压层插值 - ICON模式"""
    # 自动检测维度名称
    for name in ['level', 'level_full', 'lev']:
        if name in data_var.dims:
            height_name = name
            break
    
    # 找到上下边界层
    level_above = (data_pfull > plvl_target).argmax(dim=height_name).compute()
    level_below = xr.where(level_above - 1 < 0, 0, level_above - 1)
    
    # 获取气压值
    value_above = data_pfull.isel({height_name: level_above})
    value_below = data_pfull.isel({height_name: level_below})
    
    # 线性插值
    f = xr.where(
        np.abs(value_above - value_below) < 1e-5, 0.5,
        (plvl_target - value_below) / (value_above - value_below)
    )
    f = f.clip(0, 1)
    
    data_interp = (1-f) * data_var.isel({height_name: level_below}) + f * data_var.isel({height_name: level_above})
    data_interp = data_interp.expand_dims(dim={"plev": [plvl_target]}, axis=-2)
    data_interp['plev'].attrs = {'units': unit, 'long_name': 'pressure level'}
    
    return data_interp

In [22]:
import pandas as pd
from tqdm.auto import tqdm

# 配置参数
TARGET_PRESSURE_LEVELS = [250, 500, 850]  # hPa
OUTPUT_DIR = './processed_data/divergence_3d/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 模式数据字典
model_data = {
    'cntl': {
        'ua': ua_cntl,
        'va': va_cntl,
        'pfull': pfull_cntl
    },
    'p4k': {
        'ua': ua_p4k,
        'va': va_p4k,
        'pfull': pfull_p4k
    },
    '4co2': {
        'ua': ua_4co2,
        'va': va_4co2,
        'pfull': pfull_4co2
    }
}

print("="*70)
print("🚀 开始批量处理三个模式的散度计算")
print("="*70)
print(f"📊 目标气压层: {TARGET_PRESSURE_LEVELS} hPa")
print(f"📁 输出目录: {OUTPUT_DIR}")
print("="*70)

🚀 开始批量处理三个模式的散度计算
📊 目标气压层: [250, 500, 850] hPa
📁 输出目录: ./processed_data/divergence_3d/


In [23]:
def interpolate_and_calc_divergence(ua, va, pfull, target_plev_hpa, dx, dy):
    """插值并计算散度"""
    ua_interp = hacky_plvl_interpolation(ua, pfull / 100, target_plev_hpa)
    va_interp = hacky_plvl_interpolation(va, pfull / 100, target_plev_hpa)
    div = numpy_divergence(ua_interp.squeeze(), va_interp.squeeze(), dx, dy)
    
    return xr.DataArray(
        div, dims=('time', 'lat', 'lon'),
        coords={'time': ua_interp.time, 'lat': ua_interp.lat, 'lon': ua_interp.lon, 'plev': target_plev_hpa},
        attrs={'units': '1/s', 'long_name': f'Divergence at {target_plev_hpa} hPa'}
    )

def calculate_divergence_for_model(model_name, ua, va, pfull, target_plevs):
    """计算多个气压层的散度"""
    print(f"\n{'='*50}\n{model_name.upper()}\n{'='*50}")
    dx, dy = compute_grid_spacing(ua.lat.values, ua.lon.values)
    
    divergence_list = []
    for plev in target_plevs:
        print(f"  Processing {plev} hPa...")
        div_da = interpolate_and_calc_divergence(ua, va, pfull, plev, dx, dy)
        divergence_list.append(div_da)
    
    ds = xr.concat(divergence_list, dim='plev').to_dataset(name='divergence')
    ds.attrs = {'model': model_name, 'pressure_levels': f'{target_plevs} hPa'}
    return ds

In [24]:
# 批量处理
import pandas as pd
results = {}
start_time = time.time()

for model_name, data in model_data.items():
    try:
        ds_div = calculate_divergence_for_model(
            model_name, data['ua'], data['va'], data['pfull'], TARGET_PRESSURE_LEVELS
        )
        
        output_file = os.path.join(OUTPUT_DIR, f'divergence_{model_name}.nc')
        ds_div.to_netcdf(output_file, mode='w')
        results[model_name] = ds_div
        print(f"  ✅ Saved: {output_file}\n")
    except Exception as e:
        print(f"  ❌ Error: {e}\n")

print(f"\n{'='*50}")
print(f"✅ Complete! Time: {(time.time()-start_time)/60:.1f} min")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"{'='*50}")


CNTL
  Processing 250 hPa...


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 391.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


  Processing 500 hPa...


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 391.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Res

  Processing 850 hPa...


...

  Processing 500 hPa...


sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 391.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large g

  Processing 850 hPa...


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 391.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Resource temporarily unavailable
sh: fork: retry: Res

  ✅ Saved: ./processed_data/divergence_3d/divergence_4co2.nc


✅ Complete! Time: 6.6 min
📁 Output: ./processed_data/divergence_3d/


## 快速查看结果

In [25]:
# 查看结果
for model in model_data.keys():
    f = os.path.join(OUTPUT_DIR, f'divergence_{model}.nc')
    if os.path.exists(f):
        ds = xr.open_dataset(f)
        print(f"{model.upper()}: {ds.plev.values} hPa | Shape: {dict(ds.divergence.sizes)}")
    else:
        print(f"{model.upper()}: File not found")

CNTL: [250 500 850] hPa | Shape: {'plev': 3, 'time': 5114, 'lat': 15, 'lon': 167}
P4K: [250 500 850] hPa | Shape: {'plev': 3, 'time': 5114, 'lat': 15, 'lon': 167}
4CO2: [250 500 850] hPa | Shape: {'plev': 3, 'time': 5114, 'lat': 15, 'lon': 167}


In [31]:
xr.open_dataset(r'/work/mh1498/m301257/code/processed_data/divergence_3d/divergence_cntl.nc')

<xarray.Dataset> Size: 307MB
Dimensions:     (plev: 3, time: 5114, lat: 15, lon: 167)
Coordinates:
  * plev        (plev) int64 24B 250 500 850
  * time        (time) datetime64[ns] 41kB 1980-01-01 1980-01-02 ... 1993-12-31
  * lat         (lat) float64 120B -14.0 -12.0 -10.0 -8.0 ... 8.0 10.0 12.0 14.0
  * lon         (lon) float64 1kB 0.0 2.0 4.0 6.0 ... 352.0 354.0 356.0 358.0
Data variables:
    divergence  (plev, time, lat, lon) float64 307MB ...
Attributes:
    model:            cntl
    pressure_levels:  [250, 500, 850] hPa